# 01 · Matriz de confusão

**Bloco 1 do workshop.**

Pergunta que este notebook responde: *que tipo de erro o modelo comete?*

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# dataset canônico do workshop — NÃO altere estes parâmetros
X, y = make_classification(n_samples=20000, n_features=20, n_informative=8,
                           n_redundant=4, weights=[0.99, 0.01], flip_y=0.0,
                           class_sep=1.5, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
# split extra para calibrar no notebook 03 (nunca calibre no teste)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.25,
                                              stratify=y_tr, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f"teste: {len(y_te)} casos | {int(y_te.sum())} fraudes | prevalência {y_te.mean():.2%}")

## 1.1 O modelo burro

Antes de olhar o modelo real, meça o baseline que não faz nada.

In [ ]:
pred_burro = np.zeros_like(y_te)
print(f"acurácia do modelo que sempre diz 'legítima': {(pred_burro == y_te).mean():.4f}")
print(f"fraudes detectadas: {int(((pred_burro == 1) & (y_te == 1)).sum())} de {int(y_te.sum())}")

## 1.2 A matriz do modelo real, no limiar padrão

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             balanced_accuracy_score, matthews_corrcoef)

pred = (proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()

print(f"VN={tn:>5}   FP={fp:>5}")
print(f"FN={fn:>5}   VP={tp:>5}")
print()
print(f"Acurácia:          {(tp+tn)/len(y_te):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_te, pred):.4f}")
print(f"MCC:               {matthews_corrcoef(y_te, pred):.4f}")
print()
print(classification_report(y_te, pred, digits=3))

### ✏️ Tarefa 1 — traduzir os erros

Escreva na célula abaixo o que **FP** e **FN** significam no **seu** desafio, não neste dataset de fraude.

Critério: a frase precisa nomear uma pessoa e uma consequência. "Falso positivo é quando o modelo erra ao dizer positivo" não vale.

In [ ]:
FP_significa = "..."   # preencha
FN_significa = "..."   # preencha

print("FP:", FP_significa)
print("FN:", FN_significa)

### ✏️ Tarefa 2 — acurácia vs balanced accuracy

Por que os dois números são tão diferentes? Responda em texto na célula abaixo.

In [ ]:
resposta_2 = """
...
"""
print(resposta_2)

### ✏️ Tarefa 3 — precisão ou recall?

Para o seu problema, qual das duas é a métrica primária? Justifique pelo custo do erro, não pelo que costuma aparecer em tutorial.

In [ ]:
metrica_primaria = "..."   # "precisão" ou "recall"
porque = "..."

print(metrica_primaria, "—", porque)

## 1.3 Fatiamento por subgrupo

Métrica agregada esconde falha localizada. Aqui criamos um subgrupo artificial só para exercitar o código. **O valor real está em fazer isso no seu modelo, no notebook 05.**

In [ ]:
grupo = np.where(X_te[:, 0] > 0, "A", "B")

for g in np.unique(grupo):
    m = grupo == g
    tn_, fp_, fn_, tp_ = confusion_matrix(y_te[m], pred[m], labels=[0, 1]).ravel()
    rec = tp_/(tp_+fn_) if (tp_+fn_) else float('nan')
    print(f"grupo {g}: n={m.sum():>5}  positivos={int(y_te[m].sum()):>3}  recall={rec:.3f}")

### ✏️ Tarefa 4 — qual é o seu subgrupo?

Nomeie um recorte relevante do seu domínio: região, canal, faixa de valor, período, versão do app, idioma. Sempre existe um.

In [ ]:
subgrupo_relevante = "..."
porque_importa = "..."
print(subgrupo_relevante, "—", porque_importa)